In [ ]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
from persiantools.jdatetime import JalaliDate

def jalali_to_gregorian(jalali_str):
    y, m, d = map(int, jalali_str.split('/'))
    return JalaliDate(y, m, d).to_gregorian()

project_candidates = [Path.cwd(), *Path.cwd().parents]
PROJECT_DIR = next((path for base in project_candidates for path in (base, base / 'commodity' / 'copper') if (path / 'data' / 'raw' / 'physical').is_dir()), None)
if PROJECT_DIR is None:
    raise FileNotFoundError('Could not locate commodity/copper from the current directory')
csv_path = PROJECT_DIR / 'data' / 'raw' / 'physical' / 'copper_cathode_physical_raw.csv'

df = pd.read_csv(csv_path)
df = df[df["date"].notna()].copy()

# Convert Jalali dates to Gregorian dates.
df["trade_date"] = df["date"].astype(str).str.strip().apply(jalali_to_gregorian)
df["trade_date"] = pd.to_datetime(df["trade_date"])

df = df.sort_values("trade_date").drop_duplicates("trade_date")
df["gap_days"] = df["trade_date"].diff().dt.days.fillna(0).astype(int)

# Plot observed trading dates.
plt.figure(figsize=(14, 3))
plt.scatter(df["trade_date"], [1] * len(df), s=20, color="tab:blue")
plt.yticks([])
plt.xlabel("تاریخ معامله")
plt.title("روزهای معامله (تاریخ‌های موجود در فایل)")
plt.grid(axis="x", alpha=0.25)

# Display gaps longer than one day.
gaps = df[df["gap_days"] > 1]
for _, row in gaps.iterrows():
    plt.axvline(row["trade_date"], color="red", alpha=0.3)

plt.tight_layout()
plt.show()

# Optional visual markers for gaps.
print("تعداد روزهای معاملاتی:", len(df))
print("بزرگترین فاصله بین معاملات:", df["gap_days"].max(), "روز")
print("شکاف‌های بزرگ‌تر از 1 روز:")
print(gaps[["trade_date", "gap_days"]])

In [ ]:
from pathlib import Path

import jdatetime
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


# Locate the project from the current directory or a parent workspace.
project_candidates = [Path.cwd(), *Path.cwd().parents]
PROJECT_DIR = next((path for base in project_candidates for path in (base, base / 'commodity' / 'copper') if (path / 'data' / 'raw' / 'physical').is_dir()), None)
if PROJECT_DIR is None:
    raise FileNotFoundError('Could not locate commodity/copper from the current directory')

PHYSICAL_PATH = (
    PROJECT_DIR
    / "data"
    / "raw"
    / "physical"
    / "copper_cathode_physical_raw.csv"
)

BUBBLE_PATH = (
    PROJECT_DIR
    / "data"
    / "processed"
    / "bubble"
    / "copper_certificate_bubble.csv"
)

# Validate input paths.
print("Physical:", PHYSICAL_PATH, PHYSICAL_PATH.exists())
print("Bubble:", BUBBLE_PATH, BUBBLE_PATH.exists())

if not PHYSICAL_PATH.exists():
    raise FileNotFoundError(PHYSICAL_PATH)

if not BUBBLE_PATH.exists():
    raise FileNotFoundError(BUBBLE_PATH)
# Set True to retain only common coverage across all three series.
ONLY_OVERLAP = True


# -----------------------------
# 1. Read physical-market data.
# -----------------------------
physical = pd.read_csv(
    PHYSICAL_PATH,
    encoding="utf-8-sig",
    low_memory=False,
)

numeric_columns = [
    "Price",
    "ArzeBasePrice",
    "Quantity",
    "arze",
]

for column in numeric_columns:
    physical[column] = pd.to_numeric(
        physical[column],
        errors="coerce",
    )


def jalali_to_gregorian(value):
    """Convert YYYY/MM/DD Jalali text to pandas Timestamp."""
    try:
        year, month, day = map(
            int,
            str(value).replace("-", "/").split("/")[:3],
        )
        gregorian = jdatetime.date(
            year,
            month,
            day,
        ).togregorian()

        return pd.Timestamp(gregorian)

    except (ValueError, TypeError):
        return pd.NaT


physical["date_gregorian"] = physical["date"].apply(
    jalali_to_gregorian
)

physical = physical.dropna(
    subset=["date_gregorian"]
).copy()


# ------------------------------------------
# 2. Construct the daily volume-weighted traded price.
# ------------------------------------------
traded = physical.loc[
    physical["Price"].gt(0)
    & physical["Quantity"].gt(0)
].copy()

traded["price_x_quantity"] = (
    traded["Price"] * traded["Quantity"]
)

daily_trade_price = (
    traded.groupby(
        "date_gregorian",
        as_index=False,
    )
    .agg(
        price_x_quantity=(
            "price_x_quantity",
            "sum",
        ),
        traded_quantity=(
            "Quantity",
            "sum",
        ),
    )
)

daily_trade_price["Price"] = (
    daily_trade_price["price_x_quantity"]
    / daily_trade_price["traded_quantity"]
)


# -------------------------------------
# 3. Construct the daily volume-weighted offer base price.
# -------------------------------------
base = physical.loc[
    physical["ArzeBasePrice"].gt(0)
    & physical["arze"].gt(0)
].copy()

base["base_price_x_offer"] = (
    base["ArzeBasePrice"] * base["arze"]
)

daily_base_price = (
    base.groupby(
        "date_gregorian",
        as_index=False,
    )
    .agg(
        base_price_x_offer=(
            "base_price_x_offer",
            "sum",
        ),
        offered_quantity=(
            "arze",
            "sum",
        ),
    )
)

daily_base_price["ArzeBasePrice"] = (
    daily_base_price["base_price_x_offer"]
    / daily_base_price["offered_quantity"]
)


# ---------------------------------
# 4. Read the copper intrinsic-price proxy.
# ---------------------------------
intrinsic = pd.read_csv(
    BUBBLE_PATH,
    encoding="utf-8-sig",
)

intrinsic["date_gregorian"] = pd.to_datetime(
    intrinsic["date"],
    errors="coerce",
)

intrinsic["intrinsic_price_irr_per_kg"] = pd.to_numeric(
    intrinsic["intrinsic_price_irr_per_kg"],
    errors="coerce",
)

intrinsic = intrinsic.dropna(
    subset=[
        "date_gregorian",
        "intrinsic_price_irr_per_kg",
    ]
).copy()


# ---------------------------------
# 5. Merge all three series by date.
# ---------------------------------
plot_data = (
    daily_trade_price[
        ["date_gregorian", "Price"]
    ]
    .merge(
        daily_base_price[
            ["date_gregorian", "ArzeBasePrice"]
        ],
        on="date_gregorian",
        how="outer",
    )
    .merge(
        intrinsic[
            [
                "date_gregorian",
                "intrinsic_price_irr_per_kg",
            ]
        ],
        on="date_gregorian",
        how="outer",
    )
    .sort_values("date_gregorian")
)


# Optionally restrict to dates covered by all three series.
if ONLY_OVERLAP:
    valid_dates = plot_data.dropna(
        subset=[
            "Price",
            "ArzeBasePrice",
            "intrinsic_price_irr_per_kg",
        ]
    )["date_gregorian"]

    if valid_dates.empty:
        raise ValueError(
            "No common date was found between the three series."
        )

    plot_data = plot_data.loc[
        plot_data["date_gregorian"].between(
            valid_dates.min(),
            valid_dates.max(),
        )
    ].copy()


# ---------------------------------
# 6. Plot the comparison.
# ---------------------------------
fig, ax = plt.subplots(figsize=(16, 8))

ax.plot(
    plot_data["date_gregorian"],
    plot_data["Price"],
    label="Physical traded Price",
    color="#0077B6",
    linewidth=2,
    marker="o",
    markersize=3,
)

ax.plot(
    plot_data["date_gregorian"],
    plot_data["ArzeBasePrice"],
    label="Physical ArzeBasePrice",
    color="#F4A261",
    linewidth=1.8,
    marker="o",
    markersize=3,
)

ax.plot(
    plot_data["date_gregorian"],
    plot_data["intrinsic_price_irr_per_kg"],
    label="Intrinsic price (LME × USD)",
    color="#2A9D8F",
    linewidth=2,
)

ax.set_title(
    "Copper physical traded price, base offer price, and intrinsic price"
)
ax.set_xlabel("Date")
ax.set_ylabel("IRR per kg")
ax.ticklabel_format(
    axis="y",
    style="plain",
    useOffset=False,
)

ax.legend()
ax.grid(alpha=0.25)

plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Display full history without restricting to common coverage.
ONLY_OVERLAP = False

# Remove missing values separately for each series.
# This connects observed points without fabricating intermediate observations.
trade_plot = (
    daily_trade_price[
        ["date_gregorian", "Price"]
    ]
    .dropna()
    .sort_values("date_gregorian")
)

base_plot = (
    daily_base_price[
        ["date_gregorian", "ArzeBasePrice"]
    ]
    .dropna()
    .sort_values("date_gregorian")
)

intrinsic_plot = (
    intrinsic[
        [
            "date_gregorian",
            "intrinsic_price_irr_per_kg",
        ]
    ]
    .dropna()
    .sort_values("date_gregorian")
)


# Start the axis at the beginning of Jalali year 1386.
start_date = pd.Timestamp(
    jdatetime.date(
        1386,
        1,
        1,
    ).togregorian()
)

end_date = max(
    trade_plot["date_gregorian"].max(),
    base_plot["date_gregorian"].max(),
    intrinsic_plot["date_gregorian"].max(),
)


fig, ax = plt.subplots(
    figsize=(20, 9)
)

# Traded price.
ax.plot(
    trade_plot["date_gregorian"],
    trade_plot["Price"],
    label="Physical traded Price",
    color="#0077B6",
    linewidth=1.8,
    marker="o",
    markersize=2.5,
    linestyle="-",
)

# Offer base price.
ax.plot(
    base_plot["date_gregorian"],
    base_plot["ArzeBasePrice"],
    label="Physical ArzeBasePrice",
    color="#F4A261",
    linewidth=1.6,
    marker="o",
    markersize=2.5,
    linestyle="-",
)

# Intrinsic-price proxy.
ax.plot(
    intrinsic_plot["date_gregorian"],
    intrinsic_plot["intrinsic_price_irr_per_kg"],
    label="Intrinsic price (LME × USD)",
    color="#2A9D8F",
    linewidth=2,
    linestyle="-",
)

ax.set_xlim(
    start_date,
    end_date,
)

ax.set_title(
    "Copper physical traded price, base offer price, and intrinsic price"
)

ax.set_xlabel("Date")
ax.set_ylabel("IRR per kg")

ax.ticklabel_format(
    axis="y",
    style="plain",
    useOffset=False,
)

ax.grid(
    alpha=0.25,
)

ax.legend(
    loc="upper left",
)

plt.xticks(
    rotation=45,
)

plt.tight_layout()
plt.show()